In [ ]:
!pip install tensorrt ultralytics

import torch
import tensorrt as trt

print("GPU Aktif mi?:", torch.cuda.is_available())
print("CUDA Versiyonu:", torch.version.cuda)
print("TensorRT Versiyonu:", trt.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 636.2 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.0 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-11.2.1.2-py3-none-any.whl size=16537 sha256=bc91e858556698dc69a5aec352ac820ba4ac06a42abf230e43199da870dc1615
  Stored in directory: /root/.cache/pip/wheels/74/d7/1e/ef122bffd3247ebd1bd40aa1b521e2aa7aa6bf6d4c38931254
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-11.2.1.2-py3-none-any.whl size=23048 s

In [ ]:
import zipfile
import os

zip_path = '/content/baseline_images.zip'
extract_to = '/content/baseline_images'
os.makedirs(extract_to, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Zip dosyası açıldı!")
    print("Açılan klasördeki dosya sayısı:", len(os.listdir(extract_to)))
except Exception as e:
    print(f"Zip açılırken sorun oluştu: {e}")

Zip dosyası açıldı!
Açılan klasördeki dosya sayısı: 2


In [ ]:
import cv2
import glob
import os

IMAGE_DIR = "/content/baseline_images"  # Elinizdeki görsellerin bulunduğu klasör
OUTPUT_VIDEO = "/content/test_thermal_video.mp4"
FPS = 10  # Saniyede kaç kare geçsin (isteğinize göre ayarlayın)

# Görselleri sıralı şekilde çek
image_paths = sorted(
    glob.glob(f"{IMAGE_DIR}/**/*.[jJ][pP][gG]", recursive=True) +
    glob.glob(f"{IMAGE_DIR}/**/*.[pP][nN][gG]", recursive=True)
)

if not image_paths:
    print(f"{IMAGE_DIR} klasöründe görsel bulunamadı!")
else:
    # İlk görselden boyut bilgisini al
    first_frame = cv2.imread(image_paths[0])
    height, width, _ = first_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, FPS, (width, height))

    print(f"{len(image_paths)} adet görsel videoya dönüştürülüyor...")

    for path in image_paths:
        img = cv2.imread(path)
        # Boyut uyuşmazlığı varsa ilk görselin boyutuna getir
        if (img.shape[0], img.shape[1]) != (height, width):
            img = cv2.resize(img, (width, height))
        out.write(img)

    out.release()
    print(f" Video oluşturuldu: {OUTPUT_VIDEO}")

584 adet görsel videoya dönüştürülüyor...
 Video oluşturuldu: /content/test_thermal_video.mp4


In [ ]:
import cv2
import time
import numpy as np
from ultralytics import YOLO

#Real-Time Edge Video Pipeline

class EdgeVideoPipeline:
    def __init__(self, engine_path, min_aspect_ratio=0.4, max_human_pixel=254, temp_threshold=40.0):
        #TensorRT INT8 Engine yükleniyor
        self.model = YOLO(engine_path, task="detect")
        self.min_aspect_ratio = min_aspect_ratio
        self.max_human_pixel = max_human_pixel
        self.temp_threshold = temp_threshold
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

    def process_frame(self, gray_frame):
      #1.Ön İşleme: CLAHE + Inferno renk paleti
      enhanced_gray = self.clahe.apply(gray_frame)
      enhanced_frame = cv2.applyColorMap(enhanced_gray, cv2.COLORMAP_INFERNO)

      # TensorRT INT8 Çıkarımı(Inference)
      results = self.model(
            enhanced_gray,
            conf=0.35,        #0.25 varsyaılan değeri yerine 0.35 yaparak zayıf/çift kutuları ele
            iou=0.45,         #0.70 varsayılan değeri yerine 0.45 yaparak çakışan kutuları agresifce birleştir
            verbose=False
      )[0]

      for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            w, h = x2 - x1, y2 - y1

            if w <= 0 or h <= 0:
                continue

            # Geometri Filtresi (Aspect Ratio)
            if (h / float(w)) < self.min_aspect_ratio:
                continue

            roi = enhanced_gray[y1:y2, x1:x2]
            if roi.size == 0 or np.max(roi) > self.max_human_pixel:
                continue

            # Sıcaklık Analizi (%90 Percentile)
            top_10 = np.percentile(roi, 90)
            avg_heat = np.mean(roi[roi >= top_10])
            temp = round(float(30.0 + (avg_heat * 0.03)), 1)
            is_anomaly = temp >= self.temp_threshold

            # Çizim
            color = (0, 0, 255) if is_anomaly else (0, 255, 0)
            cv2.rectangle(enhanced_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(enhanced_frame, f"Insan: {temp}C", (x1, max(y1-5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

      return enhanced_frame

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Canlı Video Akış Similasyon Testi

VIDEO_PATH = "/content/test_thermal_video.mp4" # Test için yükleyeceğiniz video
OUTPUT_VIDEO_PATH = "/content/processed_thermal_video_fps5.mp4"
ENGINE_PATH = "/content/yolov8_gold_best_int8.engine"

pipeline = EdgeVideoPipeline(engine_path=ENGINE_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"Video açılamadı: {VIDEO_PATH}")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    PLAYBACK_FPS =5.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, PLAYBACK_FPS, (width, height))

    frame_count = 0
    start_time = time.time()

    print("Edge video pipline çalışıyor...")

    while cap.isOpened():
        ret,frame = cap.read()
        if not ret:
          break

        #Gri tonlamaya çevir (Termal girdi kabülü)
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if len(frame.shape) == 3 else frame

        #Pipeline işleme
        processed_frame = pipeline.process_frame(gray_frame)

        #Anlık işleme FPS bilgilerini ekrana basma
        frame_count += 1
        elapsed_time = time.time() - start_time
        current_fps = frame_count / elapsed_time if elapsed_time > 0 else 0

        cv2.putText(processed_frame, f"Edge processing FPS: {current_fps:.1f}", (20, 40),
                      cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(processed_frame)

    cap.release()
    out.release()
    print(f"Video kaydedildi: {OUTPUT_VIDEO_PATH}")
    print(f"Ortalama canlı işleme hızı {current_fps:.1f} FPS")



Edge video pipline çalışıyor...
Loading /content/yolov8_gold_best_int8.engine for TensorRT inference...
Video kaydedildi: /content/processed_thermal_video_fps5.mp4
Ortalama canlı işleme hızı 15.2 FPS
